In [ ]:
# Install requirements

!pip install gradio fastapi uvicorn chromadb python-pptx python-multipart PyMuPDF requests

In [1]:
# Imports

import os
import fitz  # PyMuPDF
import gradio as gr
import chromadb
from chromadb.utils import embedding_functions
from pptx import Presentation
from pptx.util import Inches, Pt
from dotenv import load_dotenv
import google.generativeai as genai

In [2]:
# Load API Key
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

genai.configure(api_key=GOOGLE_API_KEY)
gemini_model = genai.GenerativeModel("gemini-1.5-flash")


In [3]:
# PDF Extraction

def extract_pdf(path):
    doc = fitz.open(path)
    pages = [page.get_text("text") for page in doc]
    title = doc.metadata.get("title", "")
    authors = doc.metadata.get("author", "")
    return {
        "title": title if title else "Untitled",
        "authors": authors if authors else "Unknown",
        "pages": pages,
    }

In [4]:
def chunk_text(pages, chunk_size=1000, overlap=200):
    text = "\n".join(pages)
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append({
            "text": text[start:end],
            "page_range": f"{start//chunk_size+1}"
        })
        start += chunk_size - overlap
    return chunks

In [5]:
# Embeddings + ChromaDB

client = chromadb.PersistentClient(path="chroma_store")
collection = client.get_or_create_collection("papers")

def embed_and_index(chunks, paper_id):
    for i, ch in enumerate(chunks):
        collection.add(
            ids=[f"{paper_id}_{i}"],
            documents=[ch["text"]],
            metadatas=[{"paper_id": paper_id, "page_range": ch["page_range"]}],
        )

def semantic_search(query, k=5):
    results = collection.query(query_texts=[query], n_results=k)
    return results["documents"][0] if results["documents"] else []

In [7]:
def call_gemini(prompt, context=""):
    """
    Calls Gemini model with prompt + context.
    Returns generated text.
    """
    full_prompt = f"{prompt}\n\nContext:\n{context}"
    response = gemini_model.generate_content(full_prompt)
    return response.text

In [8]:
def build_pptx(meta, summary, bullets, related, filename="output.pptx"):
    prs = Presentation()

    # Title slide
    slide_layout = prs.slide_layouts[0]
    slide = prs.slides.add_slide(slide_layout)
    title = slide.shapes.title
    subtitle = slide.placeholders[1]
    title.text = meta["title"]
    subtitle.text = f"Authors: {meta['authors']}"

    # Summary slide
    slide_layout = prs.slide_layouts[1]
    slide = prs.slides.add_slide(slide_layout)
    slide.shapes.title.text = "Summary"
    content = slide.placeholders[1]
    content.text = summary

    # Bullets slide
    slide = prs.slides.add_slide(slide_layout)
    slide.shapes.title.text = "Key Points"
    content = slide.placeholders[1]
    content.text = "\n".join(bullets)

    # Related works
    slide = prs.slides.add_slide(slide_layout)
    slide.shapes.title.text = "Related Works"
    content = slide.placeholders[1]
    content.text = "\n".join(related)

    prs.save(filename)
    return filename

In [9]:
def process_pdf(path):
    meta = extract_pdf(path)
    chunks = chunk_text(meta["pages"])
    embed_and_index(chunks, paper_id=meta["title"])

    # Retrieval context for summarization
    top_context = semantic_search("summarize", k=5)
    context_text = "\n\n".join(top_context)

    # 🔹 Gemini prompts
    summary_prompt = """You are an expert researcher.
    Summarize the paper into:
    1) TL;DR (1 sentence)
    2) 4 concise bullets: problem, approach, key result, one limitation
    3) 2 follow-up research questions."""
    
    bullets_prompt = """Extract 4–6 bullet points (problem, approach, results, limitations)."""
    
    related_prompt = """List 3 related works or research directions, short rationale each."""

    # 🔹 Call Gemini
    summary = call_gemini(summary_prompt, context_text)
    bullets = call_gemini(bullets_prompt, context_text).split("\n")
    related = call_gemini(related_prompt, context_text).split("\n")

    # 🔹 Build slide deck
    pptx_path = build_pptx(meta, summary, bullets, related)

    return {
        "tldr": summary,
        "bullets": [b.strip() for b in bullets if b.strip()],
        "related": [r.strip() for r in related if r.strip()],
        "pptx_path": pptx_path,
    }


In [11]:
# Gradio UI

with gr.Blocks(theme="default") as demo:
     with gr.Row():
        with gr.Column():
            gr.HTML("<h1 style='margin: 0; font-size: 2.2em; color: #2563eb; font-weight: 800; text-align:center;'>📑<b>DigestAI</b></h1>")
            gr.HTML("<p style='font-size: 1.2em; color: #374151; margin-top: 4px; text-align:center;'>Crafting websites with AI magic</p>")

    with gr.Column(scale=1):
        file_in = gr.File(label="📤 Upload PDF", file_types=[".pdf"])
        btn = gr.Button("⚡ Process Paper", variant="primary")

    with gr.Column(scale=2):
        with gr.Tab("📑 Summary"):
            summary_out = gr.Textbox(label="TL;DR", lines=3, interactive=False)

        with gr.Tab("📝 Key Points"):
            bullets_out = gr.Textbox(label="Key Insights", lines=10, interactive=False)

        with gr.Tab("🔍 Related Works"):
            related_out = gr.Textbox(label="Similar Papers", lines=8, interactive=False)

        with gr.Tab("🎞 Slide Deck"):
            pptx_out = gr.File(label="Download Slide Deck")

    # Backend connection
    def run(file):
        res = process_pdf(file.name)
        return res["tldr"], "\n".join(res["bullets"]), "\n".join(res["related"]), res["pptx_path"]

    btn.click(fn=run, inputs=file_in, outputs=[summary_out, bullets_out, related_out, pptx_out])

demo.launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
